In [18]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 

In [19]:
df = pd.read_csv("/Users/sherry/Desktop/HACKATHON/zomato_kpt_dataset_final_Sherry.csv")

In [20]:
print(df.columns)

Index(['order_time', 'restaurant_id', 'cuisine_type', 'restaurant_capacity',
       'chef_count', 'efficiency_score', 'total_items',
       'item_complexity_score', 'active_orders_in_kitchen', 'queue_length',
       'weather', 'festival_flag', 'weekend', 'prep_time', 'chef_load_ratio',
       'queue_per_chef', 'capacity_utilization', 'effective_capacity',
       'effective_load_ratio', 'restaurant_avg_prep_time',
       'restaurant_std_prep_time', 'prep_time_recent_avg',
       'prep_time_recent_std', 'complexity_load_interaction',
       'complexity_queue_interaction', 'is_weekend', 'hourly_avg_load',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'log_queue_length',
       'log_active_orders', 'cuisine_encoded'],
      dtype='object')


In [21]:
columns_to_drop = [

    # -------------------------------------------------
    # 🔹 Raw Congestion (redundant after ratios)
    # -------------------------------------------------
    "active_orders_in_kitchen",
    "queue_length",
    "capacity_utilization",
    "log_active_orders",
    "hourly_avg_load",

    # -------------------------------------------------
    # 🔹 Derived Capacity (redundant)
    # -------------------------------------------------
    "restaurant_capacity",
    "effective_capacity",

    # -------------------------------------------------
    # 🔹 Redundant Complexity
    # -------------------------------------------------
    "total_items",
    "complexity_queue_interaction",

    # -------------------------------------------------
    # 🔹 Redundant Restaurant Variance
    # -------------------------------------------------
    "restaurant_std_prep_time",
    "prep_time_recent_std",

    # -------------------------------------------------
    # 🔹 Raw Time Columns
    # -------------------------------------------------
    "hour",
    "day_of_week",
    "is_weekend",

    # -------------------------------------------------
    # 🔹 Weak / Non-causal
    # -------------------------------------------------
    "restaurant_rating",

    # -------------------------------------------------
    # 🔹 IDs (not for modeling)
    # -------------------------------------------------
    "restaurant_id",
    "order_time"
]

In [22]:
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

In [23]:
df.columns

Index(['cuisine_type', 'chef_count', 'efficiency_score',
       'item_complexity_score', 'weather', 'festival_flag', 'weekend',
       'prep_time', 'chef_load_ratio', 'queue_per_chef',
       'effective_load_ratio', 'restaurant_avg_prep_time',
       'prep_time_recent_avg', 'complexity_load_interaction', 'hour_sin',
       'hour_cos', 'dow_sin', 'dow_cos', 'log_queue_length',
       'cuisine_encoded'],
      dtype='object')

In [24]:


corr = df.select_dtypes(include=np.number).corr().abs()

high_corr = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)

print(high_corr[high_corr > 0.85])

chef_load_ratio           effective_load_ratio    0.994986
restaurant_avg_prep_time  prep_time_recent_avg    0.881877
chef_load_ratio           log_queue_length        0.878213
queue_per_chef            log_queue_length        0.877985
chef_load_ratio           queue_per_chef          0.876009
queue_per_chef            effective_load_ratio    0.865093
effective_load_ratio      log_queue_length        0.864142
dtype: float64


In [25]:
df = df.drop(columns=["effective_load_ratio"], errors="ignore")

In [26]:
df = df.drop(columns=["cuisine_type"], errors="ignore")

In [27]:
df["weather"] = df["weather"].astype("category").cat.codes

In [28]:
df.columns

Index(['chef_count', 'efficiency_score', 'item_complexity_score', 'weather',
       'festival_flag', 'weekend', 'prep_time', 'chef_load_ratio',
       'queue_per_chef', 'restaurant_avg_prep_time', 'prep_time_recent_avg',
       'complexity_load_interaction', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'log_queue_length', 'cuisine_encoded'],
      dtype='object')

In [29]:
df.to_csv("zomato_kpt_final_19_features_Sherry.csv", index=False)

In [31]:
sample_df = df.sample(n=100000, random_state=42)

sample_df.to_csv("zomato_kpt_sample_for_GitHub.csv", index=False)